<a href="https://colab.research.google.com/github/Iyke-7/Iyke-7/blob/main/SQL_Analytics_Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import pandas as pd
import io

# This will prompt you to select a file from your computer
uploaded = files.upload()

# Read the uploaded file into a Pandas DataFrame
# (Replace 'your_file.csv' with the actual name of your file)
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

Saving swiggy.csv to swiggy.csv


In [3]:
# Display the first 5 rows
df.head()

,ID,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time
0,211,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59
1,221,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56
2,246,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50
3,248,Indiranagar,Bangalore,New Punjabi Hotel,250.0,3.9,500,"North Indian,Punjabi,Tandoor,Chinese",80 Feet Road,57
4,249,Indiranagar,Bangalore,Nh8,350.0,4.0,50,"Rajasthani,Gujarati,North Indian,Snacks,Desser...",80 Feet Road,63


# Step-1 Data Cleaning queries

In [4]:
import sqlite3

# Create an in-memory SQLite database and load the dataframe into a table named 'swiggy'
conn = sqlite3.connect(':memory:')
df.to_sql('swiggy', conn, index=False, if_exists='replace')

8680

In [5]:
query = """
SELECT COUNT(*) FROM swiggy;
"""

result = pd.read_sql(query, conn)
print(result)

   COUNT(*)
0      8680


In [6]:
# 1. Print your actual DataFrame columns to check their spelling
print("Actual columns in your DataFrame:", df.columns.tolist())

# 2. Optional: Clean up column names (strip trailing spaces and make them lowercase)
df.columns = df.columns.str.strip().str.lower()

# 3. Re-load the cleaned dataframe into SQLite
df.to_sql('swiggy', conn, index=False, if_exists='replace')

Actual columns in your DataFrame: ['ID', 'Area', 'City', 'Restaurant', 'Price', 'Avg ratings', 'Total ratings', 'Food type', 'Address', 'Delivery time']


8680

In [7]:
query = """
SELECT
  SUM(CASE WHEN Restaurant IS NULL THEN 1 ELSE 0 END) AS null_names,
  SUM(CASE WHEN "Avg ratings" IS NULL THEN 1 ELSE 0 END) AS null_ratings,
  SUM(CASE WHEN City IS NULL THEN 1 ELSE 0 END) AS null_cities
FROM swiggy;
"""

result = pd.read_sql(query, conn)
print(result)

   null_names  null_ratings  null_cities
0           0             0            0


In [8]:
# 1. Use cursor.execute for DELETE/UPDATE/INSERT statements
# 2. Wrap "Avg ratings" in double quotes because it contains a space
query = """
DELETE FROM swiggy
WHERE "Avg ratings" = 'NEW' OR "Avg ratings" = '--';
"""

# Execute the delete statement
conn.execute(query)
conn.commit()  # Save the changes to the database

print("Rows deleted successfully!")


Rows deleted successfully!


In [9]:

query = """
UPDATE swiggy
SET "Avg ratings" = CAST("Avg ratings" AS REAL)
WHERE "Avg ratings" NOT IN ('NEW', '--');
"""

# 3. Execute and commit
conn.execute(query)
conn.commit()

print("Done! Ratings standardized.")

Done! Ratings standardized.


# Step 2 — Analysis queries

In [10]:
# Q1: How many restaurants are listed per city?
query = """
SELECT City, COUNT(*) AS RestaurantCount
FROM swiggy
GROUP BY City
ORDER BY RestaurantCount DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
print(result)


        city  RestaurantCount
0    Kolkata             1346
1     Mumbai             1277
2    Chennai             1106
3       Pune             1090
4  Hyderabad             1075
5  Bangalore              946
6  Ahmedabad              717
7      Delhi              611
8      Surat              512


In [14]:
# Q2: What are the most popular Food type across India?
query = """
SELECT "Food type", COUNT(*) AS FoodCount
FROM swiggy
GROUP BY "Food type"
ORDER BY FoodCount DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
print(result)

              food type  FoodCount
0                Indian        389
1               Chinese        277
2          North Indian        246
3             Fast Food        240
4          South Indian        228
5        Indian,Chinese        106
6              Desserts        106
7  North Indian,Chinese         99
8                Bakery         90
9             Beverages         82


In [15]:
# Q3: Top 5 cities with highest average restaurant rating?
query = """
SELECT City, AVG("Avg ratings") AS AvgRating
FROM swiggy
GROUP BY City
ORDER BY AvgRating DESC
LIMIT 5;
"""

result = pd.read_sql(query, conn)
print(result)


        city  AvgRating
0    Chennai   3.776311
1  Bangalore   3.762896
2  Hyderabad   3.701395
3    Kolkata   3.698811
4     Mumbai   3.599374


In [17]:
# Q4: What is the average price for two across cities?
query = """
SELECT City, ROUND(AVG(CAST(REPLACE(Price, ',', '') AS INTEGER)), 0) AS AvgCost
FROM swiggy
WHERE Price IS NOT NULL
GROUP BY City
ORDER BY AvgCost DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
print(result)

        city  AvgCost
0     Mumbai    394.0
1  Bangalore    383.0
2    Kolkata    362.0
3    Chennai    356.0
4       Pune    354.0
5      Delhi    333.0
6  Ahmedabad    318.0
7  Hyderabad    300.0
8      Surat    270.0


In [18]:
# Q5: Which Food type have the highest average rating?
query = """
SELECT "Food type",
       ROUND(AVG(CAST("Avg ratings" AS FLOAT)), 2) AS avg_rating,
       COUNT(*) AS restaurant_count
FROM swiggy
WHERE "Avg ratings" NOT IN ('NEW', '--')
GROUP BY "Food type"
HAVING COUNT(*) > 100
ORDER BY avg_rating DESC
LIMIT 10;
"""
result = pd.read_sql(query, conn)
print(result)

        food type  avg_rating  restaurant_count
0    South Indian        3.86               228
1        Desserts        3.64               106
2    North Indian        3.57               246
3       Fast Food        3.56               240
4          Indian        3.54               389
5  Indian,Chinese        3.49               106
6         Chinese        3.41               277


In [19]:
# Q6: Restaurants with rating above 4.5 and more than 1000 ratings

query = """
SELECT Restaurant, City, "Avg ratings", "Total ratings", Price
FROM swiggy
WHERE CAST("Avg ratings" AS FLOAT) >= 4.5
  AND CAST(REPLACE("Total ratings", '+', '') AS INTEGER) >= 1000
ORDER BY "Avg ratings" DESC
LIMIT 20;
"""

result = pd.read_sql(query, conn)
print(result)

                                           restaurant       city  avg ratings  \
0                              Corner House Ice Cream  Bangalore          4.7   
1                                              Mithai    Kolkata          4.7   
2                        New Kalpana Mistanna Bhander    Kolkata          4.7   
3                           Deliciae By Bunty Mahajan     Mumbai          4.6   
4                                       Mithai Mandir    Chennai          4.6   
5                                      Chennai Square    Kolkata          4.6   
6                                            Maharani    Kolkata          4.6   
7                                            Truffles  Bangalore          4.5   
8                                         Hungry Tide    Kolkata          4.5   
9                                         China Pearl  Bangalore          4.5   
10                              Cafe Niloufer Classic  Hyderabad          4.5   
11                          

In [20]:
# Q7: Business insight — which city has best value for money?
# (high rating, low cost)
query = """
SELECT City,
       ROUND(AVG(CAST("Avg ratings" AS FLOAT)), 2) AS avg_rating,
       ROUND(AVG(CAST(REPLACE(Price, ',', '') AS INTEGER)), 0) AS avg_cost
FROM swiggy
WHERE "Avg ratings" NOT IN ('NEW', '--')
GROUP BY City
HAVING COUNT(*) > 30
ORDER BY avg_rating DESC, avg_cost ASC
LIMIT 10;
"""
result = pd.read_sql(query, conn)
print(result)

        city  avg_rating  avg_cost
0    Chennai        3.78     356.0
1  Bangalore        3.76     383.0
2  Hyderabad        3.70     300.0
3    Kolkata        3.70     362.0
4  Ahmedabad        3.60     318.0
5     Mumbai        3.60     394.0
6      Surat        3.58     270.0
7       Pune        3.55     354.0
8      Delhi        3.53     333.0
